# Orbitando el Sol

*Modelado y Simulación en Python*

Copyright 2021 Allen Downey

Licencia: [Creative Commons Atribución-No Comercial-CompartirIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [1]:
# install Pint if necessary

try:
    import pint
except ImportError:
    !pip install pint

In [2]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)
    
download('https://github.com/AllenDowney/ModSimPy/raw/master/modsim.py')

In [3]:
# import functions from modsim

from modsim import *

En un ejemplo anterior, modelamos la interacción entre la Tierra y el Sol, simulando lo que sucedería si la Tierra se detuviera en su órbita y cayera directamente hacia el Sol.

Ahora ampliemos el modelo a dos dimensiones y simulemos una revolución de la Tierra alrededor del Sol, es decir, un año.

En el perihelio, la distancia de la Tierra al Sol es de 147,09 millones de kilómetros y su velocidad es de 30.290 m/s.

In [4]:
r_0 = 147.09e9     # initial distance m
v_0 = 30.29e3      # initial velocity m/s

Aquí están las otras constantes que necesitaremos, todas con aproximadamente 4 dígitos significativos.

In [5]:
G = 6.6743e-11     # gravitational constant N / kg**2 * m**2
m1 = 1.989e30      # mass of the Sun kg
m2 = 5.972e24      # mass of the Earth kg
t_end = 3.154e7    # one year in seconds

**Ejercicio:** Coloque las condiciones iniciales en un objeto `State` con las variables `x`, `y`, `vx` y `vy`.
Cree un objeto `System` con las variables `init` y `t_end`.

In [6]:
# Solution goes here

In [7]:
# Solution goes here

**Ejercicio:** Escribe una función llamada `universal_gravitation` que tome un `State` y un `System` y devuelva la fuerza gravitacional del Sol sobre la Tierra como un `Vector`.

Pruebe su función con las condiciones iniciales; el resultado debería ser un Vector con componentes aproximados:

```
x   -3.66e+22
y   0
```

In [8]:
# Solution goes here

In [9]:
# Solution goes here

**Ejercicio:** Escriba una función de pendiente que tome una marca de tiempo, un `State` y un `System` y calcule las derivadas de las variables de estado.

Pruebe su función con las condiciones iniciales.  El resultado debe ser una secuencia de cuatro valores, aproximadamente

```
0.0, -30290.0, -0.006, 0.0
```

In [10]:
# Solution goes here

In [11]:
# Solution goes here

**Ejercicio:** Utilice `run_solve_ivp` para ejecutar la simulación.
Guarde los valores de retorno en variables llamadas `results` y `details`.

In [12]:
# Solution goes here

Puede utilizar la siguiente función para trazar los resultados.

In [13]:
from matplotlib.pyplot import plot

def plot_trajectory(results):
    x = results.x / 1e9
    y = results.y / 1e9

    make_series(x, y).plot(label='orbit')
    plot(0, 0, 'yo')

    decorate(xlabel='x distance (million km)',
             ylabel='y distance (million km)')

In [14]:
plot_trajectory(results)

Probablemente verás que la Tierra no termina donde empezó, como esperamos que suceda después de un año.
Las siguientes celdas calculan el error, que es la distancia entre las posiciones inicial y final.

In [15]:
error = results.iloc[-1] - system.init
error

In [16]:
offset = Vector(error.x, error.y)
vector_mag(offset) / 1e9

El problema es que el algoritmo utilizado por `run_solve_ivp` no funciona muy bien con sistemas como este.
Hay dos maneras en que podemos mejorarlo.

`run_solve_ivp` toma un argumento de palabra clave, `rtol`, que especifica la "tolerancia relativa", que determina el tamaño de los pasos de tiempo en la simulación.  Los valores más bajos de `rtol` requieren pasos más pequeños, lo que produce resultados más precisos.
El valor predeterminado de `rtol` es `1e-3`.  

**Ejercicio:** Intente ejecutar la simulación nuevamente con valores más pequeños, como `1e-4` o `1e-5`, y vea qué efecto tiene en la magnitud de `offset`.

La otra forma de mejorar los resultados es utilizar un algoritmo diferente.  `run_solve_ivp` toma un argumento de palabra clave, `method`, que especifica qué algoritmo debe usar.  El valor predeterminado es `RK45`, que es un buen algoritmo de propósito general, pero no particularmente bueno para este sistema.  Una de las otras opciones es `RK23`, que suele ser menos preciso que `RK45` (con el mismo tamaño de paso), pero para este sistema resulta irrazonablemente bueno, [por razones que no entiendo del todo](https://mathoverflow.net/questions/314940/celestial-mechanics-and-runge-kutta-methods).
Otra opción más es 'DOP853', que es particularmente buena cuando `rtol` es pequeño.

**Ejercicio:** Ejecute la simulación con uno de estos métodos y vea qué efecto tiene en los resultados.  Para tener una idea de cuán eficientes son los métodos, muestre `details.nfev`, que es el número de veces que `run_solve_ivp` llamó a la función de pendiente.

In [17]:
details.nfev

## Animación

Puede utilizar la siguiente función de dibujo para animar los resultados, si desea ver cómo se ve la órbita (no en tiempo real).

In [18]:
xlim = results.x.min(), results.x.max()
ylim = results.y.min(), results.y.max()

def draw_func(t, state):
    x, y, vx, vy = state
    plot(x, y, 'b.')
    plot(0, 0, 'yo')
    decorate(xlabel='x distance (million km)',
             ylabel='y distance (million km)',
             xlim=xlim,
             ylim=ylim)

In [19]:
# animate(results, draw_func)